# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipson-mishra/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The Rule:** A page is a candidate for review if it has high visibility (at least 500 impressions) but its Click-Through Rate (CTR) is significantly lower than the median for its current ranking position tier. High-visibility pages with low CTR represent "low-hanging fruit": users are seeing the page in search results, but the title, snippet, or intent-match is failing to capture the click. We rank these by an `opportunity_score` which weights the CTR gap by the square root of impressions to prioritize high-volume impact.

**Reason Codes:**
* `sufficient_volume`: The page meets the minimum 500-impression floor for reliable measurement.
* `below_position_expected_ctr`: The page's CTR is below the median for its position tier (e.g., Top 3, Page 1).
* `visible_position`: The page is on the first two pages of search results (avg position <= 20).
* `engagement_context`: The page has at least 30 sessions, providing enough data to check on-page behavior.

In [ ]:
import os
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

def load_token():
    for path in [Path.cwd() / '.env', Path.cwd().parent / '.env', Path.cwd().parent.parent / '.env']:
        if path.exists():
            for line in path.read_text().splitlines():
                if line.strip().startswith('HF_TOKEN='):
                    return line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))
    return os.environ.get('HF_TOKEN')

token = load_token()
if not token:
    raise RuntimeError('HF_TOKEN is required through .env or the environment.')
con = duckdb.connect()
con.execute('CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN ?)', [token])
FACT = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
CONTENT = 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
MIN_IMPRESSIONS = 500
print('HF warehouse connection configured; token value is not displayed.')

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
query = f"""
WITH daily AS (
  SELECT content_hash_id, client_hash_id,
         SUM(COALESCE(gsc_impressions, 0)) AS impressions,
         SUM(COALESCE(gsc_clicks, 0)) AS clicks,
         SUM(COALESCE(gsc_sum_position, 0)) AS sum_position,
         SUM(COALESCE(ga4_sessions, 0)) AS sessions,
         -- Derive a simple proxy label: did clicks in the second half of the month decline vs the first half?
         SUM(CASE WHEN report_date >= '2026-03-16' THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_h2,
         SUM(CASE WHEN report_date < '2026-03-16' THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_h1
  FROM read_parquet('{FACT}')
  WHERE gsc_data_available IS TRUE
  GROUP BY content_hash_id, client_hash_id
)
SELECT d.*, c.content_type, c.main_intent, c.word_count,
       d.clicks * 100.0 / NULLIF(d.impressions, 0) AS ctr,
       d.sum_position * 1.0 / NULLIF(d.impressions, 0) AS avg_position,
       CASE WHEN d.clicks_h2 < d.clicks_h1 THEN 1 ELSE 0 END AS is_declining
FROM daily d
LEFT JOIN read_parquet('{CONTENT}') c USING (content_hash_id)
WHERE d.impressions >= {MIN_IMPRESSIONS}
"""
queue = con.sql(query).df()
queue['position_tier'] = pd.cut(queue['avg_position'], bins=[0, 3, 10, 20, 50, float('inf')], labels=['top_3', 'page_1', 'striking', 'page_3_5', 'deep'], right=True).astype('string').fillna('no_data')
queue = queue[queue['position_tier'] != 'no_data'].copy()

# Calculate position-tier references
reference = queue.groupby('position_tier', observed=True)['ctr'].agg(expected_ctr='median', reference_n='size')
queue = queue.join(reference, on='position_tier')

# Apply the scoring logic
queue['ctr_gap'] = (queue['expected_ctr'] - queue['ctr']).clip(lower=0)
queue['opportunity_score'] = queue['ctr_gap'] * (1 + queue['impressions']).pow(0.5)
queue['below_position_reference'] = queue['ctr'] < queue['expected_ctr']

# Construct reason codes
queue['reason_codes'] = 'sufficient_volume|'
queue.loc[queue['below_position_reference'], 'reason_codes'] += 'below_position_expected_ctr|'
queue.loc[queue['avg_position'] <= 20, 'reason_codes'] += 'visible_position|'
queue.loc[queue['sessions'] >= 30, 'reason_codes'] += 'engagement_context|'
queue['reason_codes'] = queue['reason_codes'].str.rstrip('|')

queue['suggested_action'] = queue['below_position_reference'].map({True: 'review_title_snippet_and_intent', False: 'monitor'})
queue = queue.sort_values(['opportunity_score', 'impressions'], ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

out_cols = ['rank', 'content_hash_id', 'client_hash_id', 'opportunity_score', 'ctr_gap', 'ctr', 'expected_ctr', 'reference_n', 'position_tier', 'avg_position', 'impressions', 'clicks', 'sessions', 'content_type', 'main_intent', 'reason_codes', 'suggested_action', 'is_declining']
output = queue[out_cols]
output_dir = Path('..') / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'baseline_action_score.csv'
output.to_csv(output_path, index=False)
print(f'Wrote {len(output):,} ranked rows to {output_path}')

## 3. Evaluation: Precision@K

*How well does the baseline predict our proxy label (is_declining)?*

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = output['is_declining'].mean()
p20 = precision_at_k(output['opportunity_score'], output['is_declining'], 20)
p50 = precision_at_k(output['opportunity_score'], output['is_declining'], 50)
p100 = precision_at_k(output['opportunity_score'], output['is_declining'], 100)

print(f'Base rate (declining pages): {base_rate:.3f}')
print(f'Precision@20: {p20:.3f}')
print(f'Precision@50: {p50:.3f}')
print(f'Precision@100: {p100:.3f}')

## 4. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = output.head(20).copy()
top20['confidence_note'] = top20.apply(lambda r: 'higher confidence: volume floor and reference n >= 50' if r['reference_n'] >= 50 and r['impressions'] >= 1000 else 'review carefully: borderline evidence', axis=1)
top20['what_would_make_it_wrong'] = 'Position mix, seasonality, or search intent may explain the observed gap.'
print(top20[['rank', 'opportunity_score', 'ctr', 'expected_ctr', 'position_tier', 'impressions', 'reason_codes', 'suggested_action', 'confidence_note', 'what_would_make_it_wrong']].to_string(index=False))

## 5. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
weak = output[(output['rank'] <= 20) & ((output['reference_n'] < 50) | (output['impressions'] < 1000))]
print(f'Potential weak picks in top 20: {len(weak)}')
print(weak[['rank', 'impressions', 'reference_n', 'ctr', 'expected_ctr', 'position_tier']].to_string(index=False))

# Leakage Check: ensure no forbidden columns or future-peeking fields are used in scoring
forbidden = {'health_score', 'priority_score', 'action_type', 'trend_direction', 'trend_pct', 'url', 'query', 'title'}
feature_names = {'impressions', 'clicks', 'sum_position', 'sessions', 'engaged_sessions', 'ctr', 'avg_position', 'content_type', 'main_intent', 'word_count'}
assert not (feature_names & forbidden)
assert output['rank'].is_unique and output['rank'].min() == 1
assert output['impressions'].ge(MIN_IMPRESSIONS).all()
assert output['content_hash_id'].notna().all()
print('Leakage, ranking, volume, and public-safe output checks passed.')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.